# Script 1 and 2 - Search Strategies Comparison

#### Objective
This notebook indexes CV data into a qdrant vector database using 3 different search strategies stored in separate in-memory collections:
1. **BM25**
2. **Vector Embedding**: It's a dense model
3. **SPLADE**: It's an sparse model
4. **Hybrid (dense + sparse) with Reciprocal Rank fusion**
5. **Hybrid (dense + sparse) with Relative Score Fusion**
6. **Hybrid (dense + bm25) with Reciprocal Rank fusion**
7. **Hybrid (dense + bm25) with Relative Score Fusion**

Search quality across all 7 strategies is evaluated against the ground truth matrix using Hit Rate, MRR@2, MRR@3, Precision, Recall, and NDCG@10.

Note: Since the sparse model and the bm25 algorithm occupy the same search domain (exact or near-exact term matching), combining them is nearly redundant and adds unnecessary complexity.

### 1. Initializing a local client and creating the collections

Initializing Qdrant in  mode for fast testing.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating the collections for the different search strategies

In [2]:
# DENSE_MODEL = {
#     "size": 1024,
#     "model_name": "microsoft/harrier-oss-v1-0.6b",
# }

# Harrier is too heavy, so testing with a simple one
DENSE_MODEL = {
    "size": 384,
    "model_name": "avsolatorio/GIST-all-MiniLM-L6-v2",
}
BM25_MODEL = "Qdrant/bm25"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"
COLLECTION_NAME = "search_strategies"

client.create_collection(
    collection_name=COLLECTION_NAME,
    # Defining semantic search strategies
    vectors_config={
        "dense": models.VectorParams(
            size=DENSE_MODEL["size"],
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        # BM25 and lexical search strategies use IDF (Inverse Document Frequency). 
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
        # Neural sparse embedding models (e.g. SPLADE) do not need IDF
        "sparse": models.SparseVectorParams()
    }
)

True

### 2. Loading CV Data and embedding

In [3]:
import os
import json

# Load CV data
with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

Getting dense vector (Vector embedding)

In [4]:
from sentence_transformers import SentenceTransformer, SparseEncoder
from dotenv import load_dotenv

load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

dense_model = SentenceTransformer(DENSE_MODEL["model_name"], token=HUGGING_FACE_API_KEY)
sparse_model = SparseEncoder("naver/splade-v3")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 57095.82it/s]


In [5]:
from chonkie import RecursiveChunker

chunker = RecursiveChunker(
    tokenizer=sparse_model.tokenizer, 
    chunk_size=400, # splade is based on BERT, which has 512 tokens max
)

data_points = []

def is_less_than_max_tokens(ref_model, text, applicant_name, chunk_num):
    num_tokens = len(ref_model.tokenizer.encode(text))
    max_tokens = ref_model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

def serialize_cv_for_embedding(cv: dict) -> str:
    """Formats CV text specifically for optimal embedding model vectorization."""
    certifications = ", ".join(cv.get("certifications", [])) or "None"
    skills = ", ".join(cv.get("skills", [])) or "None"
    languages = ", ".join(cv.get("languages", [])) or "None"
    
    experience = "\n".join([f"- {exp}" for exp in cv.get("experience", [])])
    education = "\n".join([f"- {edu}" for edu in cv.get("education", [])])

    text = f"""Candidate Name: {cv.get("name")}
Profession: {cv.get("profession")}
Seniority Level: {cv.get("seniority_level")}
Location: {cv.get("location")}
Years of Experience: {cv.get("experience_years")}
Languages: {languages}

About Me:
{cv.get("about_me", "")}

Skills:
{skills}

Certifications:
{certifications}

Work Experience:
{experience}

Education:
{education}
""".strip()
    
    return text

for cv in cv_data:
    cv_text = serialize_cv_for_embedding(cv)
    chunks = chunker(cv_text)
    for c in chunks:
        chunk_str = c.text if hasattr(c, "text") else str(c)
        data_points.append({
            "chunk": chunk_str,
            "cv": cv
        })

chunks = [point["chunk"] for point in data_points]
dense_embeddings = dense_model.encode(chunks).tolist()
sparse_embeddings = sparse_model.encode_document(
    chunks,
    convert_to_sparse_tensor=True # Tensor with no zeroes
).coalesce()



Transforming the sparse embedding into a Qdrant format

In [6]:
doc_indices = sparse_embeddings.indices()[0].tolist()   # Which chunk?
token_indices = sparse_embeddings.indices()[1].tolist() # Which token?
weights = sparse_embeddings.values().tolist()           # The SPLADE score

qdrant_sparse_vectors = [
    {"indices": [], "values": []} for _ in range(len(chunks))
]

for doc_id, token_id, weight in zip(doc_indices, token_indices, weights):
    qdrant_sparse_vectors[doc_id]["indices"].append(token_id)
    qdrant_sparse_vectors[doc_id]["values"].append(weight)

sparse_embeddings = [
    models.SparseVector(indices=vec["indices"], values=vec["values"])
    for vec in qdrant_sparse_vectors
]

### 3. Uploading data points and payload fields to Qdrant Collection

In [7]:
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            vector={
                "dense": dense_embeddings[idx],
                "bm25": models.Document(text=data_point["chunk"], model=BM25_MODEL, options={}),
                "sparse": sparse_embeddings[idx]
            
            },
            payload={
                **data_point["cv"],
                "chunk": data_point["chunk"]
            }
        )
        for idx, data_point in enumerate(data_points)
    ]
)   

# Index payload fields. this allows to later search by any
#   of these fields so that multiple chunks belonging to the
#   same candidate CV are grouped together into a single top
#   result per candidate.
fields_to_index = ["location", "seniority_level", "email"]
for field in fields_to_index:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=models.PayloadSchemaType.TEXT
    )


/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_51094/3899118600.py:27: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


### 4. Loading Job Descriptions & Ground Truth Evaluation Matrix

In [8]:
with open("../data/job_descriptions_train_batch.json", "r", encoding="utf-8") as f:
    job_descriptions = json.load(f)

with open("../data/ground_truth_matrix.json", "r", encoding="utf-8") as f:
    ground_truth_matrix = json.load(f)

In [9]:
job_description = 'Senior hybrid role in Medellín ($5M–7M COP/month) with 5+ years experience. Responsible for designing, deploying, and maintaining autonomous AI agents and low-code digital workflows using n8n, LangGraph, CrewAI, MCP, RAG, Python, and LLMs (GPT-4, Claude 3.5, Gemini). Connects APIs with legacy CRMs (Bitrix24, Zoho) and establishes productized SaaS formulas. Requires fluent technical English.'

### 5. Comparing Search Strategies Metrics

In [51]:
# Clear Qdrant's leftover model state from previous failed runs
client._model_embedder._batch_accumulator.clear()

result = client.query_points_groups(
    collection_name=COLLECTION_NAME,
    prefetch=None,
    query=dense_model.encode(job_description).tolist(),
    using="dense",
    limit=10,
    group_by="email",
    group_size=1
)
for rank, hit in enumerate(result.groups):
    print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")

1. {'name': 'Juan Pablo Mesa López', 'email': 'mesax1@gmail.com', 'linkedin': 'https://www.linkedin.com/in/juan-pablo-mesa-l%C3%B3pez-1633b8148', 'phone': '+57 305 239 01 67', 'about_me': 'Ph.D. in Engineering in AI, VRP, and data-driven decisions. Backend Software Engineer focusing on AI and LLM integrations with more than 5 years of experience as a Data scientist and developer.', 'profession': 'Consultant / SWE & AI Engineer / Data Scientist', 'seniority_level': 'Senior', 'location': 'Medellín, Colombia', 'experience_years': '5+', 'certifications': [], 'skills': ['Python', 'Linux', 'HuggingFace', 'OpenAI', 'Pinecone', 'FastAPI', 'Qdrant', 'Langchain', 'Unstructured', 'C++', 'SQL', 'Postgres', 'Pandas', 'Scikit-learn', 'Keras', 'PyTorch', 'Plotly', 'XGBoost', 'LLMs and RAG', 'Recommendation Systems', 'AI Chatbots', 'Metaheuristics', 'Vehicle Routing Heuristics', 'Machine Learning', 'Git & GitHub', 'Docker', 'GCP', 'Firebase', 'OR-tools', 'OSRM'], 'experience': ['Applied AI Engineer an

/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_51094/1709942945.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")


In [52]:
# Clear Qdrant's leftover model state from previous failed runs
client._model_embedder._batch_accumulator.clear()

sparse_embedding = sparse_model.encode_query(job_description).coalesce()
sparse_query_vec = models.SparseVector(
    indices=sparse_embedding.indices()[0].tolist(),
    values=sparse_embedding.values().tolist()
)

result = client.query_points_groups(
    collection_name=COLLECTION_NAME,
    prefetch=None,
    query=sparse_query_vec,
    using="sparse",
    limit=10,
    group_by="email",
    group_size=1
)
for rank, hit in enumerate(result.groups):
    print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")

1. {'name': 'ALEJANDRO LÓPEZ GIRALDO', 'email': 'alopezgiraldo7@gmail.com', 'linkedin': 'LinkedIn profile available', 'phone': '+57 314 779 6122', 'about_me': 'Strong problem-solving, critical thinking, communication, teamwork, adaptability, continuous learning, time management, organisational skills, attention to detail, and a creative, analytical mindset.', 'profession': 'Software Engineer and AI Engineer', 'seniority_level': 'Mid-Senior Level', 'location': 'Medellín, Colombia (with extensive remote experience)', 'experience_years': '3+ years in software and AI engineering, plus an engineering apprenticeship', 'certifications': ['Certified Tech Developer (full-time) - Training: 1568 hours (Digital House, May 2022 - March 2023)', 'DS4A (Data Science For All Colombia 2022) - Training: 375 hours (March 2022 - July 2022)', 'Fundamentals of Data Engineering (The Latam Engineer) - Training: 70 hours (November 2025 - December 2025)'], 'skills': ['Programming & Scripting: Python, Matlab, Oct

/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_51094/2625525960.py:20: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")


In [53]:
# Clear Qdrant's leftover model state from previous failed runs
client._model_embedder._batch_accumulator.clear()

result = client.query_points_groups(
    collection_name=COLLECTION_NAME,
    prefetch=None,
    query=models.Document(text=job_description, model=BM25_MODEL),
    using="bm25",
    limit=10,
    group_by="email",
    group_size=1
)
for rank, hit in enumerate(result.groups):
    print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")

1. {'name': 'Benjamín Herrera Madrigal', 'email': 'Benjahermad@gmail.com', 'linkedin': '', 'phone': '(+506) 8580-1397', 'about_me': 'Expert developer specializing in Artificial Intelligence (AI) and process automation with over 1 year of professional experience on the n8n platform. Possesses extensive technical knowledge in creating and optimizing complex workflows, utilizing advanced tools like n8n and CrewAI to solve business challenges through intelligent and scalable solutions. Continuously updates skills through specialized courses, focused on developing innovative technological solutions that drive operational efficiency and business competitiveness.', 'profession': 'AI and Automation Specialist Developer', 'seniority_level': 'Entry-level', 'location': '', 'experience_years': '1+ years', 'certifications': ['Professional Certification in Automation with n8n (2024)'], 'skills': ['Development of Artificial Intelligence (AI) solutions', 'Advanced business process automation', 'Profes

/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_51094/2385544968.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(f"{rank + 1}. {hit.hits[0].dict()["payload"]}\n")
